In [1]:
import pandas as pd
import os
import io
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

import tensorflow as tf
try: [tf.config.experimental.set_memory_growth(gpu, True) for gpu in tf.config.experimental.list_physical_devices("GPU")]
except: pass

from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

from mltu.preprocessors import ImageReader
from mltu.transformers import ImageResizer, LabelIndexer, LabelPadding, ImageShowCV2
from mltu.augmentors import RandomBrightness, RandomRotate, RandomErodeDilate, RandomSharpen
from mltu.annotations.images import CVImage

from mltu.tensorflow.dataProvider import DataProvider
from mltu.tensorflow.losses import CTCloss
from mltu.tensorflow.callbacks import Model2onnx, TrainLogger
from mltu.tensorflow.metrics import CERMetric, WERMetric

from model import train_model
from configs import ModelConfigs
import unicodedata
import numpy as np
import os, json, io
from pathlib import Path

In [2]:
# === CONFIGURE THIS TO YOUR DATASET ROOT ===
DATASET_ROOT = Path(r"D:\User\dataset")


In [3]:
def _load_jsonl(jsonl_path: Path):
    """Yield dicts from a JSONL, tolerant of UTF-8 BOM."""
    with io.open(jsonl_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

In [6]:
def load_split(dataset_root: Path, split: str):
    """
    Build [(image_path, text), ...] for a given split ('train' | 'val' | 'test').
    We trust 'text' from JSONL. For 'file', we take just the basename and join to split dir.
    """
    annot = dataset_root / split / "annotations" / "labels.jsonl"
    images_dir = dataset_root / split

    if not annot.exists():
        raise FileNotFoundError(f"Missing annotations file: {annot}")

    pairs = []
    missing = 0

    for rec in _load_jsonl(annot):
        # expected keys: file, text, lang, direction
        rel = rec.get("file", "")
        text = rec.get("text", "")

        # Use only the basename; map to our split folder
        basename = os.path.basename(rel)
        img_path = images_dir / basename

        if not img_path.exists():
            # try a couple of fallbacks just in case
            alt = images_dir / basename.replace(".jpg", ".png")
            if alt.exists():
                img_path = alt
            else:
                missing += 1
                continue

        pairs.append((str(img_path), text))

    if len(pairs) == 0:
        raise RuntimeError(f"No image-text pairs found for split '{split}'. Check paths and JSONL.")

    if missing:
        print(f"[{split}] Skipped {missing} entries with missing files.")

    return pairs

In [8]:
# ---- Load all splits ----
train_dataset = load_split(DATASET_ROOT, "train")   # list of (path, text)
val_dataset   = load_split(DATASET_ROOT, "val")
test_dataset  = load_split(DATASET_ROOT, "test")    # optional for eval

# ---- Build vocab and max length from TRAIN ONLY ----
train_texts = [t for _, t in train_dataset]
characters = sorted(set(ch for t in train_texts for ch in t))
max_length = max(len(t) for t in train_texts)

print(f"Train pairs: {len(train_dataset)} | Val pairs: {len(val_dataset)} | Test pairs: {len(test_dataset)}")
print(f"Vocab size: {len(characters)} | Max text length (train): {max_length}")
print(train_dataset[0])

Train pairs: 40 | Val pairs: 5 | Test pairs: 5
Vocab size: 56 | Max text length (train): 100
('D:\\User\\dataset\\train\\AHTD3A0446_Para2_3.png', 'كنساً أزال عنه كل أتر لشك الجمال . وتهب في @ شهر نيسان @ رياح حارة من الشرق')


In [9]:
# Create a ModelConfigs object to store model configurations
configs = ModelConfigs()

configs.vocab = "".join(characters)
configs.max_text_length = max_length

# CTC wiring
configs.blank_index = len(configs.vocab)     # padding/blank target index
configs.num_classes = configs.blank_index + 1  # model output classes

configs.save()
print(f"[CTC] vocab={len(configs.vocab)}, blank_index={configs.blank_index}, num_classes={configs.num_classes}")

[CTC] vocab=56, blank_index=56, num_classes=57


In [7]:
# ====== Use DataProviders per split (no random split now) ======
train_data_provider = DataProvider(
    dataset=train_dataset,
    skip_validation=True,
    batch_size=configs.batch_size,
    data_preprocessors=[ImageReader(CVImage)],
    transformers=[
        ImageResizer(configs.width, configs.height, keep_aspect_ratio=True),
        LabelIndexer(configs.vocab),
        LabelPadding(max_word_length=configs.max_text_length, padding_value=len(configs.vocab)),
    ],
)

val_data_provider = DataProvider(
    dataset=val_dataset,
    skip_validation=True,
    batch_size=configs.batch_size,
    data_preprocessors=[ImageReader(CVImage)],
    transformers=[
        ImageResizer(configs.width, configs.height, keep_aspect_ratio=True),
        LabelIndexer(configs.vocab),
        LabelPadding(max_word_length=configs.max_text_length, padding_value=len(configs.vocab)),
    ],
)

In [8]:
# Optional (for post-training evaluation)
test_data_provider = DataProvider(
    dataset=test_dataset,
    skip_validation=True,
    batch_size=configs.batch_size,
    data_preprocessors=[ImageReader(CVImage)],
    transformers=[
        ImageResizer(configs.width, configs.height, keep_aspect_ratio=True),
        LabelIndexer(configs.vocab),
        LabelPadding(max_word_length=configs.max_text_length, padding_value=len(configs.vocab)),
    ],
)

In [9]:
train_data_provider.augmentors = [
    RandomBrightness(),
    RandomErodeDilate(),
    RandomSharpen(),
    # RandomRotate(max_angle=2)  # optional: small rotations if desired
]

In [ ]:

# Creating TensorFlow model architecture
model = train_model(
    input_dim = (configs.height, configs.width, 3),
    output_dim=configs.num_classes, 
)

# Compile the model and print summary
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=configs.learning_rate), 
    loss=CTCloss(), 
    metrics=[
        CERMetric(vocabulary=configs.vocab),
        WERMetric(vocabulary=configs.vocab)
        ],
    run_eagerly=False
)
model.summary(line_length=110)

c:\Users\User\anaconda3\envs\hwrecog_env\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                   ┃ Output Shape              ┃          Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)             │ (None, 96, 1408, 3)       │                0 │ -                          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ lambda (Lambda)                │ (None, 96, 1408, 3)       │                0 │ input[0][0]                │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d (Conv2D)                │ (None, 96, 1408, 32)      │              896 │ lambda[0][0]               │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ batch_normalization            │ (None, 96, 1408, 32)      │              128 │ conv2d[0][0]               │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ leaky_re_lu (LeakyReLU)        │ (None, 96, 1408, 32)      │                0 │ batch_normalization[0][0]  │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)              │ (None, 96, 1408, 32)      │            9,248 │ leaky_re_lu[0][0]          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ batch_normalization_1          │ (None, 96, 1408, 32)      │              128 │ conv2d_1[0][0]             │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)              │ (None, 96, 1408, 32)      │              128 │ lambda[0][0]               │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ add (Add)                      │ (None, 96, 1408, 32)      │                0 │ batch_normalization_1[0][… │
│                                │                           │                  │ conv2d_2[0][0]             │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ leaky_re_lu_1 (LeakyReLU)      │ (None, 96, 1408, 32)      │                0 │ add[0][0]                  │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ dropout (Dropout)              │ (None, 96, 1408, 32)      │                0 │ leaky_re_lu_1[0][0]        │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)              │ (None, 48, 704, 32)       │            9,248 │ dropout[0][0]              │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ batch_normalization_2          │ (None, 48, 704, 32)       │              128 │ conv2d_3[0][0]             │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ leaky_re_lu_2 (LeakyReLU)      │ (None, 48, 704, 32)       │                0 │ batch_normalization_2[0][… │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)              │ (None, 48, 704, 32)       │            9,248 │ leaky_re_lu_2[0][0]        │
├───

 Total params: 1,611,448 (6.15 MB)

 Trainable params: 1,608,504 (6.14 MB)

 Non-trainable params: 2,944 (11.50 KB)

In [12]:
# Define callbacks
earlystopper = EarlyStopping(monitor="val_CER", patience=20, verbose=1, mode="min")
checkpoint = ModelCheckpoint(f"{configs.model_path}/model.keras", monitor="val_CER", verbose=1, save_best_only=True, mode="min")
trainLogger = TrainLogger(configs.model_path)
tb_callback = TensorBoard(f"{configs.model_path}/logs", update_freq=1)
reduceLROnPlat = ReduceLROnPlateau(monitor="val_CER", factor=0.9, min_delta=1e-10, patience=5, verbose=1, mode="auto")
# model2onnx = Model2onnx(f"{configs.model_path}/model.keras")

# Train the model
model.fit(
    train_data_provider,
    validation_data=val_data_provider,
    epochs= 25,#configs.train_epochs,
    callbacks=[earlystopper, checkpoint, trainLogger, reduceLROnPlat, tb_callback],
    # workers=configs.train_workers
)


Epoch 1/25


InvalidArgumentError: Graph execution error:

Detected at node compile_loss/CTCLoss defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\asyncio\base_events.py", line 641, in run_forever

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\asyncio\base_events.py", line 1986, in _run_once

  File "c:\Users\User\anaconda3\envs\hwrecog_env\Lib\asyncio\events.py", line 88, in _run

 